In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Mutate)=
# Mutate

*Mutating amino-acid groups in a molecular system.*

The function {func}`molsysmt.build.mutate` substitutes amino acids in peptide or protein groups. A target name alone does not establish a complete chemical state or a validated side-chain conformation.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.build.mutate`.
:::


## Basic usage

Let's show how this function works by building a pentapeptide and applying residue mutations:


In [2]:
import molsysmt as msm


In [3]:
molsys = msm.build.build_peptide('TyrGlyGlyPheMet')


In [4]:
msm.info(molsys, element='group')


index,id,name,type,n atoms,component index,chain index,molecule index,molecule type,entity index,entity name
0,0,TYR,amino acid,21,0,0,0,peptide,0,peptide
1,1,GLY,amino acid,7,0,0,0,peptide,0,peptide
2,2,GLY,amino acid,7,0,0,0,peptide,0,peptide
3,3,PHE,amino acid,20,0,0,0,peptide,0,peptide
4,4,MET,amino acid,17,0,0,0,peptide,0,peptide


We now mutate glycine at position 2 to alanine using a string mutation code:


In [5]:
new_molsys = msm.build.mutate(molsys, mutations=["GLY-2-ALA"])


In [6]:
msm.info(new_molsys, element='group')


index,id,name,type,n atoms,component index,chain index,molecule index,molecule type,entity index,entity name
0,0,TYR,amino acid,22,0,0,0,peptide,0,peptide
1,1,GLY,amino acid,7,0,0,0,peptide,0,peptide
2,2,ALA,amino acid,10,0,0,0,peptide,0,peptide
3,3,PHE,amino acid,20,0,0,0,peptide,0,peptide
4,4,MET,amino acid,17,0,0,0,peptide,0,peptide


Mutations can also be specified using a dictionary mapping group index to target amino-acid name:


In [7]:
new_molsys = msm.build.mutate(molsys, mutations={1: 'ALA', 2: 'VAL'})


Or mapping source residue name to target residue type across all matching residues:


In [8]:
new_molsys = msm.build.mutate(molsys, mutations={'GLY': 'ALA'}, keys='group_name')



## Inspecting reconstruction coverage

The default native engine removes non-backbone atoms in each target group and
uses the bounded {func}`molsysmt.build.add_missing_heavy_atoms` tool. Several
missing side-chain atoms, invalid anchors or conflicting local geometry remain
unassessed with a warning. A VAL or TRP target can therefore retain only its
backbone. Always inspect `msm.build.get_missing_heavy_atoms(new_molsys)` before
using a mutant in downstream calculations. Hydrogen addition and sequence labels
do not establish a complete target. Keep these diagnostics visible in your own
preparation workflow; the introductory notebook filter hides warnings.

Use `engine='PDBFixer'` explicitly if you choose that optional provider's wider
reconstruction. There is no automatic fallback. Its generated coordinates still
need independent geometric assessment; this function does not optimize rotamers,
resolve clashes or minimize energy. The source system remains unchanged.


## Restricting mutation targets

`selection` always selects source atoms. Selecting one atom enables its entire
group for a substitution; a list of integers contains atom indices, not group
indices. A Boolean mask has one entry per source atom. Repeated atoms do not
repeat substitutions. `syntax` applies to selection strings, including MDTraj
expressions such as `resid 1`.

Every valid request is intersected with the enabled groups, including dictionary
keys interpreted as group indices. Empty selections, empty requests and disjoint
valid requests return an independent copy without reconstruction. Unknown group
IDs and out-of-range group indices are errors. IDs are literal string labels;
selection can disambiguate repeated labels. Source-name checks and conflicts are
validated before either engine edits a working system.

Selection limits substitutions. Native heavy-atom repair also stays within the
target groups, while hydrogen completion and PDBFixer heavy repair retain their
whole-system scope. PDBFixer requires unique integer IDs in each addressed chain
and non-negative target IDs. Its PDB-text route rounds retained coordinates to
0.001 angstrom. Use the native engine for other literal group labels.


In [9]:
atom_indices = msm.select(molsys, selection='group_index==1')
new_molsys = msm.build.mutate(molsys, mutations={'GLY': 'ALA'},
                            keys='group_name', selection=atom_indices[:1])
msm.get(new_molsys, element='group', group_name=True)

['TYR', 'ALA', 'GLY', 'PHE', 'MET']

:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Build peptide <Tutorial_Build_peptide>`: Generate capped peptide structures from sequence with {func}`molsysmt.build.build_peptide`.
- {ref}`Info <Tutorial_Info>`: Display structural and topological summaries of a molecular system with {func}`molsysmt.basic.info`.
- {ref}`Mutate <Tutorial_Mutate>`: Mutate specified amino-acid groups with {func}`molsysmt.build.mutate`.
:::
